# 04 - Long Term Memory, Short vs Long Term Memory, Memory Schema

`MemorySaver` (checkpointer) gives short-term, per-thread memory.
`InMemoryStore` gives long-term memory shared across threads, keyed by a
namespace (e.g. per user) - swap in a persistent store backend for production.

In [ ]:
import os
from dotenv import load_dotenv
from langgraph.store.memory import InMemoryStore
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, MessagesState, START, END
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI

load_dotenv()
llm = ChatOpenAI(model="gpt-4o-mini")

## Memory Schema

Long-term memory is just a plain dict shape we agree to store under each user's
namespace - no class needed, e.g. `{"name": str, "preferences": list[str]}`.

In [ ]:
store = InMemoryStore()
user_namespace = ("users", "sri")

store.put(user_namespace, "profile", {"name": "Sri", "preferences": ["prefers short answers"]})
store.get(user_namespace, "profile").value

## Short vs Long Term Memory

The graph below reads long-term memory (the `store`, shared across threads) on
every turn, while the checkpointer (`MemorySaver`) tracks each thread's own
conversation separately.

In [ ]:
def chat_node(state, config, *, store):
    namespace = ("users", config["configurable"]["user_id"])
    profile = store.get(namespace, "profile")
    profile_text = profile.value if profile else {}
    system = {"role": "system", "content": f"User profile (long-term memory): {profile_text}"}
    response = llm.invoke([system] + state["messages"])
    return {"messages": [response]}

builder = StateGraph(MessagesState)
builder.add_node("chat", chat_node)
builder.add_edge(START, "chat")
builder.add_edge("chat", END)

checkpointer = MemorySaver()  # short-term: this conversation thread only
chat_app = builder.compile(checkpointer=checkpointer, store=store)  # store: long-term, shared across threads

In [ ]:
config_thread_a = {"configurable": {"thread_id": "session-a", "user_id": "sri"}}
print(chat_app.invoke({"messages": [HumanMessage(content="What do you know about me?")]}, config_thread_a)["messages"][-1].content)

# Different thread, same user -> long-term memory still carries over
config_thread_b = {"configurable": {"thread_id": "session-b", "user_id": "sri"}}
print(chat_app.invoke({"messages": [HumanMessage(content="What do you know about me?")]}, config_thread_b)["messages"][-1].content)

## Updating Long Term Memory

Write new facts back to the store as they're learned, so future threads see them too.

In [ ]:
def remember_preference(user_id, preference):
    namespace = ("users", user_id)
    profile = store.get(namespace, "profile").value
    profile["preferences"].append(preference)
    store.put(namespace, "profile", profile)

remember_preference("sri", "likes LangGraph demos")
store.get(("users", "sri"), "profile").value